In [1]:
# FruitNet - Fast GPU Preprocessing & Training Notebook
# Run this cell on Google Colab with T4 GPU enabled!

import os
import json
import shutil
import glob
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from google.colab import files
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

# 1. Setup Kaggle
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json file when prompted:")
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    if 'kaggle.json' in uploaded:
        with open('/root/.kaggle/kaggle.json', 'wb') as f:
            f.write(uploaded['kaggle.json'])
        os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip install -q kaggle
!kaggle datasets download -d shashwatwork/fruitnet-indian-fruits-dataset-with-quality -p /content/raw --unzip

# 2. Organize & Split
RAW_DIR = '/content/raw'
BASE_DIR = '/content/Fruits10'
ORGANIZED_DIR = os.path.join(BASE_DIR, 'organized')
SPLIT_DIR = os.path.join(BASE_DIR, 'splits')
METADATA_DIR = os.path.join(BASE_DIR, 'metadata')
MODEL_DIR = os.path.join(BASE_DIR, 'models')

for d in [ORGANIZED_DIR, SPLIT_DIR, METADATA_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

MAPPING = {
    os.path.join('Bad Quality_Fruits', 'Apple_Bad'): 'apple_Bad',
    os.path.join('Bad Quality_Fruits', 'Banana_Bad'): 'banana_Bad',
    os.path.join('Bad Quality_Fruits', 'Guava_Bad'): 'guava_Bad',
    os.path.join('Bad Quality_Fruits', 'Lime_Bad'): 'lime_Bad',
    os.path.join('Bad Quality_Fruits', 'Orange_Bad'): 'orange_Bad',
    os.path.join('Bad Quality_Fruits', 'Pomegranate_Bad'): 'pomegranate_Bad',
    os.path.join('Good Quality_Fruits', 'Apple_Good'): 'apple_Good',
    os.path.join('Good Quality_Fruits', 'Banana_Good'): 'banana_Good',
    os.path.join('Good Quality_Fruits', 'Guava_Good'): 'guava_Good',
    os.path.join('Good Quality_Fruits', 'Lime_Good'): 'lime_Good',
    os.path.join('Good Quality_Fruits', 'Orange_Good'): 'orange_Good',
    os.path.join('Good Quality_Fruits', 'Pomegranate_Good'): 'pomegranate_Good',
    os.path.join('Mixed Qualit_Fruits', 'Apple'): 'apple_Mixed',
    os.path.join('Mixed Qualit_Fruits', 'Banana'): 'banana_Mixed',
    os.path.join('Mixed Qualit_Fruits', 'Guava'): 'guava_Mixed',
    os.path.join('Mixed Qualit_Fruits', 'Orange'): 'orange_Mixed',
    os.path.join('Mixed Qualit_Fruits', 'Pomegranate'): 'pomegranate_Mixed',
    os.path.join('Mixed Qualit_Fruits', 'Lemon'): 'lime_Mixed',
}

data = []
for sub_rel, target_cls in MAPPING.items():
    src_folder = os.path.join(RAW_DIR, sub_rel)
    dst_folder = os.path.join(ORGANIZED_DIR, target_cls)
    os.makedirs(dst_folder, exist_ok=True)
    if os.path.exists(src_folder):
        for img_name in os.listdir(src_folder):
            src_file = os.path.join(src_folder, img_name)
            if os.path.isfile(src_file) and img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                dst_file = os.path.join(dst_folder, img_name)
                shutil.copy2(src_file, dst_file)
                data.append({'filepath': dst_file, 'class': target_cls})

df = pd.DataFrame(data)
classes = sorted(df['class'].unique().tolist())
class_to_index = {cls: idx for idx, cls in enumerate(classes)}
index_to_class = {str(idx): cls for idx, cls in enumerate(classes)}
df['label'] = df['class'].map(class_to_index)

metadata = {
    'num_classes': len(classes),
    'classes': classes,
    'class_to_index': class_to_index,
    'index_to_class': index_to_class,
    'image_size': [224, 224]
}

with open(os.path.join(METADATA_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=4)

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=(25.0/30.0), stratify=temp_df['label'], random_state=42)

train_df.to_csv(os.path.join(SPLIT_DIR, 'train.csv'), index=False)
val_df.to_csv(os.path.join(SPLIT_DIR, 'validation.csv'), index=False)
test_df.to_csv(os.path.join(SPLIT_DIR, 'test.csv'), index=False)

# 3. Train Model
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):
    image = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    return image, label

def create_dataset(dataframe, training=False):
    ds = tf.data.Dataset.from_tensor_slices((dataframe['filepath'].values, dataframe['label'].values))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(1000)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df, training=False)

base_model = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = layers.Rescaling(1./255)(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(len(classes), activation='softmax')(x)
model = models.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Starting Phase 1 Training (Transfer Learning)...")
model.fit(train_ds, validation_data=val_ds, epochs=5)

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Starting Phase 2 Training (Fine Tuning)...")
model_save_path = os.path.join(MODEL_DIR, 'fruit_quality_efficientnetb0.keras')
callbacks = [
    ModelCheckpoint(model_save_path, save_best_only=True, monitor='val_accuracy'),
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
]
model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=callbacks)

# 4. Download Trained Model & Metadata automatically
print("Auto-downloading model & metadata to your computer...")
files.download(model_save_path)
files.download(os.path.join(METADATA_DIR, 'metadata.json'))
print("DONE!")

[✓] Output logged successfully.
